In [ ]:
!pip -q install pandas numpy

import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/ReinforcementLearningCrypto")
FILES = {
    "SUI": "SUI20947-USD_DataHr.csv",
    "AAVE": "AAVE-USD_DataHr.csv",
    "ETC": "ETC-USD_DataHr.csv",
    "BTC": "BTC-USD_DataHr.csv",
    "ES":  "ES=F_DataHr.csv",
    "GC":  "GC=F_DataHr.csv",
}

def read_market_csv(path: Path) -> pd.DataFrame:
    """
    Reads the course CSV with 2 header rows (group label + ticker),
    returns a dataframe indexed by Datetime with columns:
    ['Close','High','Low','Open','Volume'] as floats.
    """
    df = pd.read_csv(path, header=[0,1], index_col=0)
    # index name is likely 'Datetime'
    df.index = pd.to_datetime(df.index, utc=True, errors="coerce")
    df = df[~df.index.isna()].sort_index()

    # Flatten multiindex columns, keep only the first level names (Close/High/Low/Open/Volume)
    # Example: ('Close','ETC-USD') -> 'Close'
    df.columns = [c[0] for c in df.columns]

    # Keep only expected columns (some files may have 'Price' col label weirdness)
    keep = [c for c in ["Close","High","Low","Open","Volume"] if c in df.columns]
    df = df[keep].copy()

    # Coerce to numeric
    for c in keep:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop rows with no Close
    df = df.dropna(subset=["Close"])

    return df

data = {}
for k, fname in FILES.items():
    path = DATA_DIR / fname
    data[k] = read_market_csv(path)
    print(k, data[k].index.min(), "->", data[k].index.max(), "rows:", len(data[k]))

SUI 2024-03-06 00:00:00+00:00 -> 2026-02-23 23:00:00+00:00 rows: 17246
AAVE 2024-03-06 00:00:00+00:00 -> 2026-02-23 23:00:00+00:00 rows: 17246
ETC 2024-03-06 00:00:00+00:00 -> 2026-02-23 23:00:00+00:00 rows: 17245
BTC 2024-03-06 00:00:00+00:00 -> 2026-02-23 23:00:00+00:00 rows: 17248
ES 2024-03-06 05:00:00+00:00 -> 2026-02-24 03:00:00+00:00 rows: 11211
GC 2024-03-06 05:00:00+00:00 -> 2026-02-24 03:00:00+00:00 rows: 11256


In [ ]:
# Use crypto union time range
start = min(data[x].index.min() for x in ["SUI","AAVE","ETC","BTC"])
end   = max(data[x].index.max() for x in ["SUI","AAVE","ETC","BTC"])

full_index = pd.date_range(start=start, end=end, freq="H", tz="UTC")
print("Full hourly index:", full_index[0], "->", full_index[-1], "len:", len(full_index))

Full hourly index: 2024-03-06 00:00:00+00:00 -> 2026-02-23 23:00:00+00:00 len: 17280


/tmp/ipykernel_419/781446516.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_index = pd.date_range(start=start, end=end, freq="H", tz="UTC")


In [ ]:
def to_hourly(df: pd.DataFrame) -> pd.DataFrame:
    # if there are duplicates or irregularities, take last within hour
    return df.resample("H").last()

for k in data:
    data[k] = to_hourly(data[k])

/tmp/ipykernel_419/2389357633.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  return df.resample("H").last()


In [ ]:
# reindex
aligned = {k: data[k].reindex(full_index) for k in data}

# Save raw availability BEFORE forward fill
ES_has_trade = aligned["ES"]["Close"].notna().astype(int)
GC_has_trade = aligned["GC"]["Close"].notna().astype(int)

In [ ]:
'''
Forward-fill ES and GC safely:
forward fill up to 6 hours (overnight gaps)
do not fill entire weekends
'''
def safe_ffill(series: pd.Series, limit: int = 6) -> pd.Series:
    return series.ffill(limit=limit)

for k in ["ES", "GC"]:
    for col in ["Close", "High", "Low", "Open", "Volume"]:
        if col in aligned[k].columns:
            aligned[k][col] = safe_ffill(aligned[k][col], limit=6)

In [ ]:
# Create core returns series (Close → log return)
def log_return(close: pd.Series) -> pd.Series:
    return np.log(close).diff()

rets = {}
for k in ["SUI","AAVE","ETC","BTC","ES","GC"]:
    rets[k] = log_return(aligned[k]["Close"])

## Construct features (for each target crypto)

Per target (SUI/AAVE/ETC):

*   r_1h (target return)

*   mom_24h (24h sum of returns)

*   mom_4h

*   vol_24h (24h std)

*   dd_7d (drawdown from rolling 7d max)

*   other cryptos’ 1h returns (2 features)

*   BTC return

*   ES return

*   GC return

*   time-of-day sin/cos (optional but easy, good for report)

In [ ]:
def rolling_sum(x: pd.Series, window: int) -> pd.Series:
    return x.rolling(window=window, min_periods=window).sum()

def rolling_std(x: pd.Series, window: int) -> pd.Series:
    return x.rolling(window=window, min_periods=window).std()

def drawdown(close: pd.Series, window: int) -> pd.Series:
    roll_max = close.rolling(window=window, min_periods=window).max()
    return close / roll_max - 1.0

In [ ]:
def build_features(target: str) -> pd.DataFrame:
    assert target in ["SUI", "AAVE", "ETC"]

    feat = pd.DataFrame(index=full_index)

    # target series aligned
    close_t = aligned[target]["Close"].reindex(full_index)
    r_t = rets[target].reindex(full_index)

    feat[f"{target}_r1h"] = r_t
    feat[f"{target}_mom24h"] = rolling_sum(r_t, 24)
    feat[f"{target}_mom4h"] = rolling_sum(r_t, 4)
    feat[f"{target}_vol4h"] = rolling_std(r_t, 4)
    feat[f"{target}_vol24h"] = rolling_std(r_t, 24)
    feat[f"{target}_dd7d"] = drawdown(close_t, 24 * 7)

    # cross-crypto
    others = [x for x in ["SUI", "AAVE", "ETC"] if x != target]
    feat[f"{others[0]}_r1h"] = rets[others[0]].reindex(full_index)
    feat[f"{others[1]}_r1h"] = rets[others[1]].reindex(full_index)

    # macro returns aligned
    feat["BTC_r1h"] = rets["BTC"].reindex(full_index)
    feat["ES_r1h"]  = rets["ES"].reindex(full_index)
    feat["GC_r1h"]  = rets["GC"].reindex(full_index)

    feat["ES_has_trade"] = ES_has_trade.reindex(full_index)
    feat["GC_has_trade"] = GC_has_trade.reindex(full_index)

    # time-of-day
    hours = feat.index.hour.values
    feat["tod_sin"] = np.sin(2*np.pi*hours/24.0)
    feat["tod_cos"] = np.cos(2*np.pi*hours/24.0)

    needed = [
        f"{target}_r1h", f"{target}_mom24h", f"{target}_vol24h", f"{target}_dd7d", f"{target}_mom4h", f"{target}_vol4h",
        "BTC_r1h", "ES_r1h", "GC_r1h",
        f"{others[0]}_r1h", f"{others[1]}_r1h"
    ]
    feat = feat.dropna(subset=needed)

    return feat


In [ ]:
feat_SUI  = build_features("SUI")
feat_AAVE = build_features("AAVE")
feat_ETC  = build_features("ETC")

print(feat_SUI.shape, feat_AAVE.shape, feat_ETC.shape)
feat_SUI.head()

(11627, 15) (11619, 15) (11617, 15)


,SUI_r1h,SUI_mom24h,SUI_mom4h,SUI_vol4h,SUI_vol24h,SUI_dd7d,AAVE_r1h,ETC_r1h,BTC_r1h,ES_r1h,GC_r1h,ES_has_trade,GC_has_trade,tod_sin,tod_cos
2024-03-12 23:00:00+00:00,0.012615,0.013323,-0.011230,0.016701,0.017065,-0.044377,0.002821,0.009610,0.003833,0.000241,-0.000786,1,1,-0.258819,0.965926
2024-03-13 00:00:00+00:00,0.019334,0.030781,0.033948,0.009763,0.017490,-0.025722,-0.007188,-0.006516,0.000156,-0.000628,0.001017,1,1,0.000000,1.000000
2024-03-13 01:00:00+00:00,0.028883,0.051694,0.057405,0.013602,0.018338,0.000000,0.010631,0.011351,0.007755,-0.000242,-0.000370,1,1,0.258819,0.965926
2024-03-13 02:00:00+00:00,-0.010243,0.033403,0.050588,0.016656,0.018462,-0.010191,0.002740,0.001946,-0.000409,-0.000435,0.000785,1,1,0.500000,0.866025
2024-03-13 03:00:00+00:00,-0.015028,0.012150,0.022945,0.021657,0.018728,-0.024954,0.023897,0.000697,0.000827,0.000290,-0.000554,1,1,0.707107,0.707107


In [ ]:
f = build_features("ETC")
print(f.index.min(), f.index.max(), f.shape)
print(f.isna().sum().sort_values(ascending=False).head(10))
print(f[[ "ETC_r1h","ETC_mom24h","ETC_vol24h","ETC_dd7d","ES_r1h","GC_r1h"]].describe())

2024-03-12 23:00:00+00:00 2026-02-23 23:00:00+00:00 (11617, 15)
ETC_r1h       0
ETC_mom24h    0
ETC_mom4h     0
ETC_vol4h     0
ETC_vol24h    0
ETC_dd7d      0
SUI_r1h       0
AAVE_r1h      0
BTC_r1h       0
ES_r1h        0
dtype: int64
            ETC_r1h    ETC_mom24h    ETC_vol24h      ETC_dd7d        ES_r1h  \
count  11617.000000  11617.000000  11617.000000  11617.000000  11617.000000   
mean      -0.000033     -0.001847      0.007805     -0.075576      0.000022   
std        0.009316      0.042061      0.004093      0.058304      0.002055   
min       -0.324641     -0.396317      0.002047     -0.370798     -0.022224   
25%       -0.003938     -0.024349      0.005188     -0.109132     -0.000503   
50%       -0.000031     -0.001706      0.006678     -0.063304      0.000000   
75%        0.004032      0.021228      0.009428     -0.031196      0.000611   
max        0.070391      0.266700      0.072121      0.000000      0.065137   

             GC_r1h  
count  11617.000000  
mean   

In [ ]:
f = build_features("ETC")

print("ES trade rate:", f["ES_has_trade"].mean())
print("GC trade rate:", f["GC_has_trade"].mean())

print("ES zero return fraction:", (f["ES_r1h"] == 0).mean())

ES trade rate: 0.9037617285013343
GC trade rate: 0.9074632004820522
ES zero return fraction: 0.11293793578376517


## Define regimes (steady / crash / surge)


*   Crash = bottom 10% of 24h return
*   Surge = top 10%
*   Steady = middle 80%

In [ ]:
def define_regime(feat: pd.DataFrame, target: str, q: float = 0.10):
    """
    Define regimes based on 24h momentum percentiles.
    """
    mom = feat[f"{target}_mom24h"]

    low_thr = mom.quantile(q)
    high_thr = mom.quantile(1 - q)

    regime = pd.Series("steady", index=feat.index)

    regime[mom <= low_thr] = "crash"
    regime[mom >= high_thr] = "surge"

    return regime, low_thr, high_thr

In [ ]:
feat_ETC["regime"], low_ETC, high_ETC = define_regime(feat_ETC, "ETC")
feat_SUI["regime"], low_SUI, high_SUI = define_regime(feat_SUI, "SUI")
feat_AAVE["regime"], low_AAVE, high_AAVE = define_regime(feat_AAVE, "AAVE")

print(feat_ETC["regime"].value_counts())

regime
steady    9293
crash     1162
surge     1162
Name: count, dtype: int64


In [ ]:
feat_ETC.groupby("regime")[f"ETC_mom24h"].describe()

,count,mean,std,min,25%,50%,75%,max
regime,,,,,,,,
crash,1162.0,-0.077078,0.037083,-0.396317,-0.082767,-0.065837,-0.055204,-0.048343
steady,9293.0,-0.001705,0.022879,-0.048338,-0.019172,-0.001706,0.016148,0.044772
surge,1162.0,0.072244,0.031253,0.044795,0.052574,0.060584,0.079440,0.266700


In [ ]:
feat_SUI.to_csv(Path(DATA_DIR) / "features_SUI.csv")
feat_AAVE.to_csv(Path(DATA_DIR) / "features_AAVE.csv")
feat_ETC.to_csv(Path(DATA_DIR) / "features_ETC.csv")

print("Saved:", "features_SUI.csv", "features_AAVE.csv", "features_ETC.csv")

Saved: features_SUI.csv features_AAVE.csv features_ETC.csv


In [ ]:
feat_AAVE.head()

,AAVE_r1h,AAVE_mom24h,AAVE_mom4h,AAVE_vol4h,AAVE_vol24h,AAVE_dd7d,SUI_r1h,ETC_r1h,BTC_r1h,ES_r1h,GC_r1h,ES_has_trade,GC_has_trade,tod_sin,tod_cos,regime
2024-03-12 23:00:00+00:00,0.002821,-0.010520,0.007190,0.005331,0.010956,-0.012641,0.012615,0.009610,0.003833,0.000241,-0.000786,1,1,-0.258819,0.965926,steady
2024-03-13 00:00:00+00:00,-0.007188,-0.019909,0.003017,0.006795,0.011025,-0.019713,0.019334,-0.006516,0.000156,-0.000628,0.001017,1,1,0.000000,1.000000,steady
2024-03-13 01:00:00+00:00,0.010631,0.004998,0.015147,0.008045,0.010875,-0.009236,0.028883,0.011351,0.007755,-0.000242,-0.000370,1,1,0.258819,0.965926,steady
2024-03-13 02:00:00+00:00,0.002740,-0.000802,0.009004,0.007300,0.010746,-0.006517,-0.010243,0.001946,-0.000409,-0.000435,0.000785,1,1,0.500000,0.866025,steady
2024-03-13 03:00:00+00:00,0.023897,0.037362,0.030081,0.013128,0.011354,0.000000,-0.015028,0.000697,0.000827,0.000290,-0.000554,1,1,0.707107,0.707107,steady
